# Superstore Sales — Exploratory Data Analysis

This notebook explores the **Gold layer** produced by the Superstore Sales Data Pipeline.  
It connects directly to the `superstore.duckdb` analytical database and runs SQL queries against the five aggregation tables.

**Prerequisite:** Run the pipeline first so the DuckDB file exists:
```bash
make run
```

---

## Notebook sections

1. Setup and database connection  
2. Regional performance  
3. Category and sub-category analysis  
4. Customer segment breakdown  
5. Monthly time-series trends  
6. Top and bottom products  
7. Discount vs profit relationship

## 1. Setup and database connection

In [ ]:
# Core libraries for data manipulation and visualisation
import duckdb                       # Embedded OLAP engine — queries the pipeline's gold tables
import pandas as pd                 # DataFrame manipulation for post-query analysis
import matplotlib.pyplot as plt     # Core plotting library
import seaborn as sns               # Statistical visualisation built on matplotlib
from pathlib import Path            # Cross-platform path handling

# Global plotting configuration for consistent styling across all charts
sns.set_theme(style="whitegrid")    # Clean white background with subtle gridlines
plt.rcParams["figure.figsize"] = (10, 5)   # Default figure size: wide, medium height
plt.rcParams["axes.titlesize"] = 13        # Title font size
plt.rcParams["axes.labelsize"] = 11        # Axis label font size

# Display options so wide DataFrames render cleanly in the notebook
pd.set_option("display.max_columns", 50)   # Show up to 50 columns before truncation
pd.set_option("display.width", 200)        # Widen the console output

In [ ]:
# Resolve the project root so the notebook works regardless of where Jupyter is launched
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Absolute path to the DuckDB file produced by the pipeline's load step
DB_PATH = PROJECT_ROOT / "database" / "superstore.duckdb"

# Open a read-only connection to avoid any accidental writes during exploration
con = duckdb.connect(str(DB_PATH), read_only=True)

# Confirm all expected tables are present before running any queries
tables = con.execute("SHOW TABLES").fetchdf()
print(f"Connected to: {DB_PATH}")
print(f"Tables available ({len(tables)}):")
tables

## 2. Regional performance

Aggregated KPIs for each US sales region — total revenue, profit, and average margin.

In [ ]:
# Query the gold-layer regional aggregation table ordered by revenue descending
region_df = con.execute("""
    SELECT
        Region,
        ROUND(total_sales,  0) AS total_sales,
        ROUND(total_profit, 0) AS total_profit,
        total_orders,
        ROUND(avg_profit_margin, 2) AS avg_profit_margin_pct
    FROM agg_sales_by_region
    ORDER BY total_sales DESC
""").fetchdf()

region_df   # Render the result as a table

In [ ]:
# Side-by-side bar chart: total sales (left) vs total profit (right) per region
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))   # 1-row, 2-column layout

# Left panel — total sales bar chart
sns.barplot(data=region_df, x="Region", y="total_sales", ax=ax1, palette="Blues_d")
ax1.set_title("Total sales by region")
ax1.set_ylabel("Total sales (USD)")
ax1.set_xlabel("")

# Right panel — total profit bar chart
sns.barplot(data=region_df, x="Region", y="total_profit", ax=ax2, palette="Greens_d")
ax2.set_title("Total profit by region")
ax2.set_ylabel("Total profit (USD)")
ax2.set_xlabel("")

plt.tight_layout()   # Prevent subplot labels from overlapping
plt.show()

## 3. Category and sub-category analysis

In [ ]:
# Top 10 sub-categories by total revenue across all regions and time periods
top_subcats = con.execute("""
    SELECT
        Category,
        "Sub-Category"                AS SubCategory,
        ROUND(total_sales, 0)         AS total_sales,
        ROUND(total_profit, 0)        AS total_profit,
        ROUND(avg_profit_margin, 2)   AS margin_pct
    FROM agg_sales_by_category
    ORDER BY total_sales DESC
    LIMIT 10
""").fetchdf()

top_subcats

In [ ]:
# Horizontal bar chart — easier to read when category names are long
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=top_subcats,                      # The top-10 sub-categories data
    x="total_sales",                       # Revenue on the x-axis
    y="SubCategory",                       # Sub-category names on the y-axis
    hue="Category",                        # Colour-code by parent category
    dodge=False,                           # Do not split bars by hue
    ax=ax,
)
ax.set_title("Top 10 sub-categories by total sales")
ax.set_xlabel("Total sales (USD)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Sub-categories that are losing money — negative total profit
loss_subcats = con.execute("""
    SELECT
        Category,
        "Sub-Category"                AS SubCategory,
        ROUND(total_sales, 0)         AS total_sales,
        ROUND(total_profit, 0)        AS total_profit,
        ROUND(avg_discount * 100, 1)  AS avg_discount_pct
    FROM agg_sales_by_category
    WHERE total_profit < 0
    ORDER BY total_profit ASC
""").fetchdf()

print(f"Sub-categories operating at a loss: {len(loss_subcats)}")
loss_subcats

## 4. Customer segment breakdown

In [ ]:
# Customer segment KPIs — one row per segment with revenue and customer counts
segment_df = con.execute("""
    SELECT
        Segment,
        total_customers,
        total_orders,
        ROUND(total_sales,   0) AS total_sales,
        ROUND(total_profit,  0) AS total_profit,
        ROUND(avg_order_value, 2)    AS avg_order_value,
        ROUND(avg_profit_margin, 2)  AS avg_margin_pct
    FROM agg_customer_segments
    ORDER BY total_sales DESC
""").fetchdf()

segment_df

## 5. Monthly time-series trends

In [ ]:
# Monthly revenue and profit time series across the full date range
trends_df = con.execute("""
    SELECT
        order_year,
        order_month,
        order_month_name,
        total_sales,
        total_profit,
        total_orders
    FROM agg_monthly_trends
    ORDER BY order_year, order_month
""").fetchdf()

# Build a proper datetime index for plotting — combines year and month into a date
trends_df["period"] = pd.to_datetime(
    trends_df["order_year"].astype(str) + "-" + trends_df["order_month"].astype(str) + "-01"
)

trends_df.head()

In [ ]:
# Dual-axis time-series chart: sales (left axis) + profit (right axis)
fig, ax1 = plt.subplots(figsize=(13, 5))

# Left axis — total sales in blue
ax1.plot(trends_df["period"], trends_df["total_sales"], color="#2E86AB", linewidth=2, label="Sales")
ax1.set_ylabel("Total sales (USD)", color="#2E86AB")
ax1.tick_params(axis="y", labelcolor="#2E86AB")
ax1.set_xlabel("Month")

# Right axis — total profit in green, sharing the same x-axis
ax2 = ax1.twinx()
ax2.plot(trends_df["period"], trends_df["total_profit"], color="#06A77D", linewidth=2, label="Profit")
ax2.set_ylabel("Total profit (USD)", color="#06A77D")
ax2.tick_params(axis="y", labelcolor="#06A77D")

plt.title("Monthly sales and profit — 2014 to 2017")
fig.tight_layout()
plt.show()

In [ ]:
# Seasonality check — average monthly revenue grouped by calendar month across all years
seasonality = (
    trends_df
    .groupby("order_month_name", sort=False)["total_sales"]
    .mean()
    .reindex(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
)

# Plot the seasonal pattern as a simple bar chart
fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(x=seasonality.index, y=seasonality.values, ax=ax, color="#4A90E2")
ax.set_title("Average monthly sales — seasonality pattern")
ax.set_ylabel("Average sales (USD)")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 6. Top and bottom products

In [ ]:
# Top 10 best-selling products by total revenue
top_products = con.execute("""
    SELECT
        "Product Name"            AS Product,
        Category,
        "Sub-Category"            AS SubCategory,
        ROUND(total_sales,  0)    AS total_sales,
        ROUND(total_profit, 0)    AS total_profit,
        total_units
    FROM agg_product_performance
    ORDER BY total_sales DESC
    LIMIT 10
""").fetchdf()

top_products

In [ ]:
# Bottom 10 products by profit — biggest loss-makers
loss_products = con.execute("""
    SELECT
        "Product Name"            AS Product,
        "Sub-Category"            AS SubCategory,
        ROUND(total_sales,  0)    AS total_sales,
        ROUND(total_profit, 0)    AS total_profit,
        ROUND(avg_discount * 100, 1) AS avg_discount_pct
    FROM agg_product_performance
    ORDER BY total_profit ASC
    LIMIT 10
""").fetchdf()

loss_products

## 7. Discount vs profit relationship

High discounts are a common cause of unprofitable sales.  
The `fact_sales` table exposes every line item, so we can inspect the relationship directly.

In [ ]:
# Pull a representative sample of the fact table to avoid over-plotting
fact_sample = con.execute("""
    SELECT
        Sales,
        Profit,
        Discount,
        profit_margin_pct,
        profit_tier
    FROM fact_sales
    USING SAMPLE 2000 ROWS
""").fetchdf()

fact_sample.head()

In [ ]:
# Scatter plot — discount on x-axis, profit margin on y-axis, coloured by profit tier
fig, ax = plt.subplots(figsize=(11, 6))

# Colour map that matches the profit-tier labels produced by the pipeline
palette = {"Loss": "#D14520", "Low": "#EF9F27", "Medium": "#4A90E2", "High": "#1D9E75"}

sns.scatterplot(
    data=fact_sample,
    x="Discount",                 # Discount rate on x-axis
    y="profit_margin_pct",        # Profit margin percentage on y-axis
    hue="profit_tier",            # Colour-code by profit tier
    palette=palette,
    alpha=0.6,                    # Partial transparency to reveal density
    ax=ax,
)

# Reference line at 0 % margin — anything below is a loss-making transaction
ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)

ax.set_title("Discount vs profit margin — loss risk rises with higher discounts")
ax.set_xlabel("Discount rate")
ax.set_ylabel("Profit margin (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Close the DuckDB connection cleanly when exploration is complete
con.close()
print("Database connection closed.")

---

## Key insights

- **Regional performance:** West and East regions dominate both revenue and profit; South and Central lag behind.
- **Category leaders:** Technology has the highest profit margins; Furniture (specifically Tables and Bookcases) tends to lose money due to heavy discounting.
- **Segments:** Consumer segment generates the most revenue; Home Office is the smallest but has competitive average order value.
- **Seasonality:** Revenue peaks in Q4 (Oct–Dec), reflecting holiday-season purchasing patterns.
- **Discount risk:** Transactions with discount rates above ~30 % are highly likely to result in negative profit margins.

These insights emerge directly from the **pipeline's gold-layer aggregations** — no ad-hoc data wrangling required in this notebook. That is the value of a well-designed analytical layer.